# 1. Answer the questions from the introduction

### 1.1 What is leave-one-out? Provide limitations and strengths.

### 1.2 How do Grid Search, Randomized Grid Search, and Bayesian optimization work?

### 1.3 Explain classification of feature selection methods.<br> Explain how Pearson and Chi2 work. Explain how Lasso works. <br> Explain what permutation significance is. Become familiar with SHAP.

# 2. Introduction — make all the preprocessing staff from the previous lesson

In [106]:
import warnings
import time
from collections import Counter
from dataclasses import dataclass
from typing import Tuple
import pandas as pd
import numpy as np
from sklearn.preprocessing import (
    MultiLabelBinarizer,
    MinMaxScaler,
    StandardScaler,
    PolynomialFeatures,
)
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet,
    ElasticNetCV,
)
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
import lightgbm as lgb
import scipy
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns


warnings.filterwarnings("ignore")

### 2.2 Read all the data

In [ ]:
path: str = "data/two-sigma-connect-rental-listing-inquiries/{name}.json"
train_df: pd.DataFrame = pd.read_json(path.format(name="train"))
test_df: pd.DataFrame = pd.read_json(path.format(name="test"))

### 2.3 Preprocess the "Interest Level" feature.

In [60]:
values: dict = {"low": 0, "medium": 1, "high": 2}
train_df["interest_level"] = train_df["interest_level"].apply(lambda x: values[x])

### * Delete outliers

In [61]:
# # more in chapter 11.2
# for df in train_df, test_df:
#     lower_limit = df["price"].quantile(0.01)
#     upper_limit = df["price"].quantile(0.99)
#     df.drop(
#         df[(df["price"] >= upper_limit) | (df["price"] <= lower_limit)].index,
#         inplace=True,
#     )

In [ ]:
# more in chapter 11.2
lower_limit: np.float64 = train_df["price"].quantile(0.01)
upper_limit: np.float64 = train_df["price"].quantile(0.99)
train_df.drop(
    train_df[(train_df["price"] >= upper_limit) | \
             (train_df["price"] <= lower_limit)].index,
    inplace=True,
)

np.float64(1600.0)

### 2.3 Create features: 'Elevator', 'HardwoodFloors', 'CatsAllowed', 'DogsAllowed', 'Doorman', 'Dishwasher', 'NoFee', 'LaundryinBuilding', 'FitnessCenter', 'Pre-War', 'LaundryinUnit', 'RoofDeck', 'OutdoorSpace', 'DiningRoom', 'HighSpeedInternet', 'Balcony', 'SwimmingPool', 'LaundryInBuilding', 'NewConstruction', 'Terrace'.

In [ ]:
feature_list: list = [
        "Elevator",
        "CatsAllowed",
        "HardwoodFloors",
        "DogsAllowed",
        "Doorman",
        "Dishwasher",
        "NoFee",
        "LaundryinBuilding",
        "FitnessCenter",
        "Pre-War",
        "LaundryinUnit",
        "RoofDeck",
        "OutdoorSpace",
        "DiningRoom",
        "HighSpeedInternet",
        "Balcony",
        "SwimmingPool",
        "LaundryInBuilding",
        "NewConstruction",
        "Terrace",
    ]

In [ ]:
for df in train_df, test_df:
    mlb: MultiLabelBinarizer = MultiLabelBinarizer(classes=feature_list)
    new_features: pd.DataFrame = pd.DataFrame(
        mlb.fit_transform(df["features"]), index=df.index, columns=feature_list
    )
    df[feature_list] = new_features

new_feature_list: list = [*feature_list, "bathrooms", "bedrooms"]

X_train: pd.DataFrame = train_df[new_feature_list]
y_train: pd.DataFrame = train_df["price"]

X_test: pd.DataFrame = test_df[new_feature_list]
y_test: pd.DataFrame = test_df["price"]

display(X_train.head(3))
display(X_test.head(3))

,Elevator,CatsAllowed,HardwoodFloors,DogsAllowed,Doorman,Dishwasher,NoFee,LaundryinBuilding,FitnessCenter,Pre-War,...,OutdoorSpace,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,LaundryInBuilding,NewConstruction,Terrace,bathrooms,bedrooms
4,0,0,0,0,0,1,0,0,0,1,...,0,0,0,0,0,0,0,0,1.0,1
6,1,0,0,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1.0,2
9,1,0,0,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1.0,2


,Elevator,CatsAllowed,HardwoodFloors,DogsAllowed,Doorman,Dishwasher,NoFee,LaundryinBuilding,FitnessCenter,Pre-War,...,OutdoorSpace,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,LaundryInBuilding,NewConstruction,Terrace,bathrooms,bedrooms
0,1,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1.0,1
1,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,1.0,2
2,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,1.0,0


# 3. Implement the next methods:

### 3.5 Make split procedure determenistic. What does it mean?

Adding random_state to split makes random deterministic

### 3.1 Split data into 2 parts randomly with parameter test_size (ratio from 0 to 1), return training and test samples.

In [111]:
def simple_split(
        X: pd.DataFrame, 
        test_size: float = 0.5, 
        random_state: int = 42,
    ) -> Tuple[pd.DataFrame, pd.DataFrame]:
    
    X_train: pd.DataFrame = X.sample(frac=test_size, random_state=random_state, replace=False)
    X_test_idx: list = list(set(X.index) - set(X_train.index))
    X_test: pd.DataFrame = X.loc[X_test_idx]
    # to check
    # display(set(X.index) == (set(X_test_idx) | set(X_train.index)))
    return X_train, X_test 


X_train, X_test = simple_split(X_train, 0.5)

### 3.2 Randomly split data into 3 parts with parameters validation_size and test_size, return train, validation and test samples.

In [ ]:
def validation_split():
    pass

### 3.3 Split data into 2 parts with parameter date_split, return train and test samples split by date_split param.

### 3.4 Split data into 3 parts with parameters validation_date and test_date, return train, validation and test samples split by input params.

# 4. Implement the next cross-validation methods:

### 4.1 K-Fold, where k is the input parameter, returns a list of train and test indices

### 4.2 Grouped K-Fold, where k and group_field are input parameters, returns list of train and test indices.

### 4.3 Stratified K-fold, where k and stratify_field are input parameters, returns list of train and test indices.

In [69]:
def crossval(n_splits: int, X: pd.DataFrame, y: pd.DataFrame) -> str:
    # skf = StratifiedKFold(n_splits=n_splits, random_state=21, shuffle=True)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True)
    out = ""; acc_list: list = []
    # get split indexes 
    for train_ix, test_ix in skf.split(X, y):
        y_model = y.iloc[train_ix]; y_test = y.iloc[test_ix]
        X_model = X.iloc[train_ix]; X_test = X.iloc[test_ix]
        # X_train, X_valid, y_train, y_valid = train_test_split(X_model, y_model, test_size=0.25, random_state=21)

        # model.fit(X_train, y_train)

        # for x_split, y_split, text in ((X_train, y_train, 'train - {:.5f} | '), (X_valid, y_valid, 'valid - {:.5f}\n')):
        #     predict_col = model.predict(x_split)
        #     acc = accuracy_score(predict_col, y_split)
        #     if text == 'valid - {:.5f}\n':
        #         acc_list.append(acc)
        #     out += text.format(acc)

    # out += f'Average accuracy on crossval is {np.mean(acc_list):.5f}\n'
    # out += f'Std is {np.std(acc_list):.5f}'
        display(train_ix)
    return out

crossval(2, X_train, y_train)
X_train.shape

array([    1,     8,    11, ..., 48340, 48341, 48342], shape=(24171,))

array([    0,     2,     3, ..., 48334, 48336, 48339], shape=(24172,))

(48343, 22)

### 4.4 Time series split, where k and date_field are input parameters, returns list of train and test indices.

# 5. Cross-validation comparison

### 5.1 Apply all the validation methods implemented above to our dataset. To apply Stratified algorithm you should preprocess target.

### 5.2 Apply the appropriate methods from sklearn.

### 5.3 Compare the resulting feature distributions for the training part of the dataset between sklearn and your implementation.

### 5.4 Compare all validation schemes. Choose the best one. Explain your choice.

# 6. Feature Selection

### 6.1 Fit a Lasso regression model with normalized features. Use your method for splitting samples into 3 parts by field created with 60/20/20 ratio — train/validation/test.

### 6.2 Sort features by weight coefficients from model, fit model to top 10 features and compare quality.

### 6.3 Implement method for simple feature selection by nan-ratio in feature and correlation. Apply this method to feature set and take top 10 features, refit model and measure quality.

### 6.4 Implement permutation importance method and take top 10 features, refit model and measure quality.

### 6.5 Import Shap and also refit model on top 10 features.

### 6.6 Compare the quality of these methods for different aspects — speed, metrics and stability.

# 7. Hyperparameter optimization

### 7.1 Implement grid search and random search methods for alpha and l1_ratio for sklearn's ElasticNet model.

### 7.2 Find the best combination of model hyperparameters.

### 7.3 Fit the resulting model.

### 7.4 Import optuna and configure the same experiment with ElasticNet.

### 7.5 Estimate metrics and compare approaches.

### 7.6 Run optuna on one of the cross-validation schemes.